Parte 10 — Del análisis a la predicción: formulación del problema (Pasos 39–42)


In [10]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Los módulos con nombre numérico se importan con importlib
ml = importlib.import_module("flows.05_ml_models")

sns.set_theme(style="whitegrid", context="notebook")

# Esta función lee el Parquet, quita las DROPPED y devuelve X, y
X, y = ml.load_dataset()      

# Verificamos la forma y la tasa de supervivencia
print(X.shape, y.mean())           

(891, 10) 0.3838383838383838


PASO 39

# Formulación del Problema de Machine Learning (Paso 39)

| Decisión | Tu respuesta y justificación |
| :--- | :--- |
| **Tipo de problema** | Clasificación binaria supervisada. (Predecimos una categoría con dos resultados posibles: 0 o 1). |
| **Variable objetivo** | `Survived` (1 = sobrevivió, 0 = falleció). |
| **Unidad de análisis y momento de predicción** | Un pasajero individual, en el momento de embarcar (antes de que ocurra el siniestro). |
| **Métrica principal** | **ROC-AUC**. <br>*Justificación:* Como el dataset está desbalanceado (solo el 38% sobrevivió), la exactitud (*accuracy*) es engañosa. El ROC-AUC evalúa la capacidad del modelo para distinguir entre clases (positivos y negativos) independientemente del umbral de decisión, lo cual es ideal aquí. |
| **Métricas secundarias** | Precisión, Recall y Exactitud. <br>*(Se usarán para entender el comportamiento específico del modelo una vez elegido el umbral).* |
| **Costo de errores** | **Falso Negativo (predecir que muere cuando sobrevive)**: Para la aseguradora, esto significa negar una póliza o cobrar una prima muy alta a alguien que era un "buen riesgo", perdiendo un cliente.<br>**Falso Positivo (predecir que sobrevive cuando muere)**: La aseguradora asume un riesgo que terminará en un pago de siniestro (pérdida financiera directa).<br>*En seguros, el Falso Positivo suele ser más costoso financieramente, pero el Falso Negativo daña la reputación.* |
| **Criterio de éxito** | Superar el baseline (tasa de supervivencia del 38%) por un margen defendible (ej. un ROC-AUC superior a 0.80). |

PASO 40 

### Paso 40: Auditoría de variables y fuga de datos

| Columna | ¿Existe antes del evento? | ¿Riesgo de fuga? | Decisión |
| :--- | :--- | :--- | :--- |
| `Survived`, `Survivor` | No (es el objetivo) | **Fuga directa** (es el futuro) | **Excluir** |
| `PassengerId`, `Ticket`, `Name` | Sí | Identificadores únicos: el modelo los memoriza pero no generaliza para nuevos pasajeros. | **Excluir** (el título del nombre ya se extrajo en `TTitulo`) |
| `NSex`, `NEmbarked`, `NCabin`, `NTitulo` | — | Duplican variables categóricas usando un orden arbitrario (ej. asignar 0 a 'female' y 1 a 'male' inventa una jerarquía). | **Excluir** (usaremos One-Hot Encoding en el Pipeline) |
| `Age` imputada | — | Estadística calculada con los datos de **todos** los pasajeros, incluyendo los de prueba (Fuga de información). | **Usar `AgeOriginal`** (el Pipeline imputará solo con Train) |
| `Cabin`, `TCabin`, `HasCabin` | Dudoso | ¿Se registró la cabina *después* del naufragio? Es más fácil registrar la cabina de un sobreviviente que de un fallecido. | **Evaluar con y sin ella** (Paso 61) |
| `Person` | Derivada de Age/Sex | Es redundante con las variables originales. | **Excluir del modelo base** (el modelo aprenderá de Age y Sex directamente) |

PASO 41 

In [2]:
import importlib
ml = importlib.import_module("flows.05_ml_models")

# 1. ¡ESTA LÍNEA ES LA QUE FALTABA! 
# Volvemos a cargar los datos en memoria (X e y)
X, y = ml.load_dataset() 

# 2. Ahora sí, dividimos los datos sin problemas
X_train, X_test, y_train, y_test = ml.split(X, y)

# 3. Verificamos que las tasas de supervivencia sean similares
print(f"Filas de Entrenamiento: {X_train.shape[0]}")
print(f"Filas de Prueba: {X_test.shape[0]}")
print(f"Tasa de supervivencia en Train: {y_train.mean():.4f}")
print(f"Tasa de supervivencia en Test:  {y_test.mean():.4f}")

Filas de Entrenamiento: 712
Filas de Prueba: 179
Tasa de supervivencia en Train: 0.3834
Tasa de supervivencia en Test:  0.3855


PASO 42 LOS BASELINES 

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

print("--- BASELINE 1: El 'Ingenuo' (DummyClassifier) ---")
# Este modelo siempre predice la clase más frecuente (en este caso, '0' = No sobrevive)
dummy = DummyClassifier(strategy='most_frequent')
# Usamos validación cruzada (cv=5) sobre el conjunto de entrenamiento
scores_dummy = cross_val_score(dummy, X_train, y_train, cv=5, scoring='accuracy')
print(f"Exactitud (Accuracy) promedio: {scores_dummy.mean():.4f}")


print("\n--- BASELINE 2: La 'Regla de Género' ---")
# Si es 'female' predice 1, si es 'male' predice 0
# (Creamos un array con las predicciones basadas en la columna Sex de X_train)
y_pred_gender = (X_train['Sex'] == 'female').astype(int)

# Calculamos las métricas sobre el conjunto de entrenamiento
acc_gender = accuracy_score(y_train, y_pred_gender)
prec_gender = precision_score(y_train, y_pred_gender)
rec_gender = recall_score(y_train, y_pred_gender)
f1_gender = f1_score(y_train, y_pred_gender)

print(f"Exactitud (Accuracy): {acc_gender:.4f}")
print(f"Precisión: {prec_gender:.4f}")
print(f"Recall (Sensibilidad): {rec_gender:.4f}")
print(f"F1-Score: {f1_gender:.4f}")

--- BASELINE 1: El 'Ingenuo' (DummyClassifier) ---
Exactitud (Accuracy) promedio: 0.6166

--- BASELINE 2: La 'Regla de Género' ---
Exactitud (Accuracy): 0.7893
Precisión: 0.7431
Recall (Sensibilidad): 0.6886
F1-Score: 0.7148


PASO 43 

In [4]:
# 1. Calcula la edad media de mujeres usando TODOS los datos (X)
# (Usamos .mean() que ignora los NaN por defecto)
media_global = X[X['Sex'] == 'female']['Age'].mean()

# 1. Calcula la edad media de mujeres usando SOLO X_train
media_train = X_train[X_train['Sex'] == 'female']['Age'].mean()

# 2. Comparamos la diferencia
diferencia = abs(media_global - media_train)

print(f"Media de edad (Mujeres) usando TODOS los datos (X): {media_global:.4f}")
print(f"Media de edad (Mujeres) usando SOLO Train (X_train): {media_train:.4f}")
print(f"Diferencia exacta: {diferencia:.4f} años")

Media de edad (Mujeres) usando TODOS los datos (X): 27.9157
Media de edad (Mujeres) usando SOLO Train (X_train): 28.1421
Diferencia exacta: 0.2264 años


PASO 44

In [ ]:
import importlib
ml = importlib.import_module("flows.05_ml_models")
importlib.reload(ml) # Recargamos para que reconozca la nueva clase

# 1. Instanciamos nuestro transformador
imputador = ml.GroupAgeImputer(group_cols=("Sex",), target="Age")

# 2. Hacemos FIT SOLO con los datos de entrenamiento
imputador.fit(X_train)

# 3. Verificamos qué aprendió
print("Medias aprendidas por grupo:")
print(imputador.group_means_)

# 4. Hacemos TRANSFORM en ambos conjuntos (Train y Test)
X_train_imputado = imputador.transform(X_train)
X_test_imputado = imputador.transform(X_test)

# 5. Verificamos que ya no haya nulos en Age
print(f"\nNulos en Age (Train) después de transformar: {X_train_imputado['Age'].isnull().sum()}")
print(f"Nulos en Age (Test) después de transformar: {X_test_imputado['Age'].isnull().sum()}")